# Capstone — Cloud Architecture RAG Assistant (portfolio)

This notebook is designed to be **portfolio credible**:
- document ingestion (your cloud/cert docs)
- chunking choices
- FAISS vector index
- retrieval with citations
- small evaluation loop
- production mapping notes (what changes in real deployment)

It intentionally supports restricted environments via local embeddings.


In [ ]:
!pip -q install langchain langchain-community langchain-text-splitters faiss-cpu numpy sentence-transformers

In [ ]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path('/content/learn RAG')
sys.path.append(str(PROJECT_ROOT))
sys.path.append(str(PROJECT_ROOT / 'src'))

from rag_core import load_text_docs, build_chunks

DOCS_DIR = PROJECT_ROOT / 'data' / 'docs'  # put your real docs here
docs = load_text_docs(DOCS_DIR)
print('docs:', len(docs))
if docs:
    print('first doc:', docs[0][0])


## 1) Chunking strategy

Start with a baseline and keep notes:
- smaller chunks: better precision, risk losing context
- larger chunks: better context, risk mixing topics

In production you typically tune this per document type.


In [ ]:
CHUNK_CHARS = 1200
OVERLAP_CHARS = 150

chunks = build_chunks(docs, chunk_chars=CHUNK_CHARS, overlap_chars=OVERLAP_CHARS)
print('chunks:', len(chunks))
print('example metadata:', chunks[0].metadata if chunks else None)


## 2) Index in FAISS (local embeddings)

If your environment allows an embedding API, you can swap it in later.


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vectorstore = FAISS.from_texts(
    texts=[c.text for c in chunks],
    embedding=embeddings,
    metadatas=[c.metadata for c in chunks],
)
retriever = vectorstore.as_retriever(search_kwargs={'k': 6})
print('index ready')


## 3) Ask with citations

For portfolio purposes, citations matter.
If LLM APIs are blocked, we’ll still return grounded snippets + citations.


In [ ]:
import re

def citations(docs):
    lines = []
    for d in docs:
        m = d.metadata or {}
        lines.append(f"- {m.get('source_id')}#chunk={m.get('chunk_index')}")
    return "\n".join(lines)

def grounded_response(question: str, docs):
    context = "\n\n".join([d.page_content for d in docs]).strip()
    if not context:
        return "I don't know based on the available documents."
    q_tokens = set(re.findall(r"[a-zA-Z]{3,}", question.lower()))
    sents = re.split(r"(?<=[.!?])\s+", context)
    scored = []
    for s in sents:
        st = set(re.findall(r"[a-zA-Z]{3,}", s.lower()))
        scored.append((len(q_tokens & st), s))
    scored.sort(key=lambda x: x[0], reverse=True)
    best = [s for score, s in scored[:6] if score > 0]
    if not best:
        best = [s for _, s in scored[:3] if s.strip()]
    return " ".join([b.strip() for b in best]).strip()

def ask(q: str):
    hits = retriever.get_relevant_documents(q)
    ans = grounded_response(q, hits)
    return ans, hits

q = "How would you design caching and what are trade-offs?"
ans, hits = ask(q)
print('Q:', q)
print('\nAnswer:\n', ans)
print('\nCitations:\n', citations(hits))


## 4) Mini evaluation loop

Use the same eval set from `data/eval/eval_set.jsonl` initially, then replace with questions from your real docs.


In [ ]:
eval_path = PROJECT_ROOT / 'data' / 'eval' / 'eval_set.jsonl'
if eval_path.exists():
    eval_items = [json.loads(line) for line in eval_path.read_text(encoding='utf-8').splitlines() if line.strip()]
else:
    eval_items = []

def contains_all(haystack: str, needles):
    h = haystack.lower()
    return all(n.lower() in h for n in needles)

rows = []
for item in eval_items:
    q = item['question']
    expected = item.get('expected_answer_contains', [])
    hits = retriever.get_relevant_documents(q)
    context = "\n\n".join([h.page_content for h in hits])
    ans = grounded_response(q, hits)
    rows.append({
        'id': item.get('id'),
        'q': q,
        'retrieval_ok': contains_all(context, expected) if expected else True,
        'answer_ok': contains_all(ans, expected) if expected else True,
    })

if rows:
    retr_acc = sum(r['retrieval_ok'] for r in rows) / len(rows)
    ans_acc = sum(r['answer_ok'] for r in rows) / len(rows)
    print('retrieval_ok:', round(retr_acc, 3))
    print('answer_ok:', round(ans_acc, 3))
    print('fails:')
    for r in rows:
        if not r['retrieval_ok'] or not r['answer_ok']:
            print(' ', r)
else:
    print('No eval set found; add JSONL rows to data/eval/eval_set.jsonl')


## 5) Production mapping (talk track)

When moving from this notebook to a real service:
- **Ingestion becomes a pipeline**: parse → clean → chunk → embed → index (async, versioned)
- **Vector DB changes**: FAISS → managed vector DB or search service (Azure AI Search / OpenSearch / Pinecone)
- **Security**: per-document ACLs and metadata filtering
- **Observability**: log retrieved chunk ids, scores, latency, and answer outcomes
- **Evaluation loop**: collect user feedback + automated checks, then tune chunking/retrieval
